# NC3 Baseline Visualizations

**Lansing Economic Ecosystem and Community Wealth Analysis**

This notebook serves as a clean, reproducible baseline for testing visualizations for the Team NC3 capstone project with the Lansing Growth Fund.

**Focus:** Estimating the local economic footprint of the Big 4 institutional actors (MSU, GM, State Government, Local/Regional Governments) using publicly available indicators.

**Data sources used here:**
- BLS LAUS (Lansing-East Lansing MSA labor force trends)
- BLS QCEW (employment and wages by ownership and industry stand-ins)
- ACS 2024 5-year (key demographic and economic indicators)
- Live Census API example (Lansing city)

All plots are intentionally simple and presentation-ready so the team can extend them into the final scorecard and dashboard.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pathlib import Path

# Styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Paths (assumes notebook is run from repository root)
DATA_DIR = Path('Data_Files')
print('Data directory exists:', DATA_DIR.exists())

## 1. Lansing MSA Unemployment Rate Trend (LAUS)

Not-seasonally-adjusted unemployment rate for the Lansing-East Lansing MSA. Useful context for overall labor market conditions affecting all Big 4 institutions.

In [ ]:
laus = pd.read_csv(DATA_DIR / 'laus_lansing_msa_2019_latest.csv')

# Focus on unemployment rate
ur = laus[laus['measure'] == 'unemployment_rate'].copy()
ur['value'] = pd.to_numeric(ur['value'], errors='coerce')
ur['date'] = pd.to_datetime(ur['year'].astype(str) + '-' + ur['period'].str[1:], format='%Y-%m')
ur = ur.dropna(subset=['value']).sort_values('date')

fig, ax = plt.subplots()
ax.plot(ur['date'], ur['value'], linewidth=2, color='#1f77b4')
ax.set_title('Lansing-East Lansing MSA Unemployment Rate (Not Seasonally Adjusted)')
ax.set_xlabel('Date')
ax.set_ylabel('Unemployment Rate (%)')
ax.axhline(y=ur['value'].mean(), color='gray', linestyle='--', alpha=0.7, label=f'Mean: {ur["value"].mean():.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Latest available rate: {ur["value"].iloc[-1]:.1f}% ({ur["date"].iloc[-1].strftime("%b %Y")})')

## 2. QCEW Employment by Ownership (2024 Annual)

Employment levels by ownership code. Note: `own_code == 2` (state government) includes public universities such as MSU. Do not double-count MSU headcount on top of this category.

**Ownership codes:**
- 0 = Total
- 1 = Federal
- 2 = State government (includes public universities)
- 3 = Local government
- 5 = Private

In [ ]:
qcew = pd.read_csv(DATA_DIR / 'qcew_2024_a_focus.csv')

# MSA totals by ownership (industry_code == 10)
msa_own = qcew[(qcew['area_name'] == 'Lansing_MSA') & (qcew['industry_code'] == '10')].copy()
msa_own = msa_own[msa_own['own_code'].isin([1, 2, 3, 5])]

ownership_labels = {1: 'Federal', 2: 'State Gov (incl. MSU)', 3: 'Local Gov', 5: 'Private'}
msa_own['ownership'] = msa_own['own_code'].map(ownership_labels)

fig, ax = plt.subplots()
bars = ax.bar(msa_own['ownership'], msa_own['annual_avg_emplvl'], color=sns.color_palette('deep', len(msa_own)))
ax.set_title('Lansing MSA Average Employment by Ownership (QCEW 2024 Annual)')
ax.set_ylabel('Average Employment')
ax.tick_params(axis='x', rotation=15)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height):,}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(msa_own[['ownership', 'annual_avg_emplvl', 'avg_annual_pay']].to_string(index=False))

## 3. Industry Stand-ins Relevant to the Big 4 (QCEW 2024)

Selected private and public industry codes used as stand-ins:
- `336` Transportation equipment (GM plant proxy)
- `611` Educational services
- `92` / `921` Public administration

In [ ]:
# Focus on key industry stand-ins in the MSA
standins = qcew[
    (qcew['area_name'] == 'Lansing_MSA') & 
    (qcew['industry_code'].isin(['10', '336', '611', '92'])) &
    (qcew['own_code'].isin([0, 2, 5]))
].copy()

standins['label'] = standins['industry_code'] + ' (own ' + standins['own_code'].astype(str) + ')'

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=standins, x='label', y='annual_avg_emplvl', hue='own_code', dodge=False, ax=ax)
ax.set_title('Employment in Key Industry Stand-ins (Lansing MSA, 2024)')
ax.set_ylabel('Average Employment')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. ACS Key Indicators — Median Household Income by Geography (2024 5-year)

Community wealth context across the tri-county area and MSA.

In [ ]:
acs = pd.read_csv(DATA_DIR / 'acs_2024_key_indicators.csv')

# Median household income (B19013)
mhi = acs[acs['table_id'] == 'B19013'].copy()
mhi = mhi[['geo_name', 'estimate']].drop_duplicates()
mhi = mhi.sort_values('estimate', ascending=False)

fig, ax = plt.subplots()
sns.barplot(data=mhi, x='estimate', y='geo_name', orient='h', palette='viridis', ax=ax)
ax.set_title('Median Household Income by Geography (ACS 2024 5-year)')
ax.set_xlabel('Median Household Income (2024 dollars)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print(mhi.to_string(index=False))

## 5. Live Census API Example — Lansing City (ACS 2024 5-year)

Direct retrieval of total population (`B01003_001E`) and median household income (`B19013_001E`) for Lansing city, Michigan (place:46000, state:26).

This demonstrates how to pull fresh ACS data via the Census API for city-level indicators that complement the county/MSA extracts already in `Data_Files/`.

In [ ]:
API_KEY = "49e79300fe1a8148a8198f3bed53e14472206afb"  # Census API key
url = "https://api.census.gov/data/2024/acs/acs5"
params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "place:46000",
    "in": "state:26",
    "key": API_KEY
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()
lansing_df = pd.DataFrame(data[1:], columns=data[0])

# Clean and display
lansing_df['B01003_001E'] = pd.to_numeric(lansing_df['B01003_001E'])
lansing_df['B19013_001E'] = pd.to_numeric(lansing_df['B19013_001E'])
print(lansing_df)

# Simple bar comparison of the two indicators (scaled for display)
fig, ax1 = plt.subplots()
ax1.bar(['Population'], lansing_df['B01003_001E'], color='#1f77b4', width=0.4)
ax1.set_ylabel('Total Population', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.bar(['Median HH Income'], lansing_df['B19013_001E'], color='#ff7f0e', width=0.4)
ax2.set_ylabel('Median Household Income ($)', color='#ff7f0e')
ax2.tick_params(axis='y', labelcolor='#ff7f0e')

plt.title('Lansing City — Population and Median Household Income (ACS 2024 5-year)')
plt.tight_layout()
plt.show()

## Next Steps for the Team

- Extend these charts with year-over-year changes, location quotients, and scorecard-style normalized metrics.
- Add MSU-specific IPEDS indicators where they do not double-count QCEW state government employment.
- Incorporate BEA GDP/personal income and IRS migration flows for wealth retention analysis.
- Expand the Census API calls to additional places, variables, or years as needed.
- Export clean figures for the partner-ready scorecard and dashboard.

*Baseline created for Team NC3 visualization testing.*